In [2]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [3]:
import folium
import pandas as pd

In [4]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

In [5]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [7]:
# Initial the map
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. 
# In addition, add Launch site name as a popup label

for idx, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']

    # Add a Circle
    folium.Circle(
        location=coordinate,
        radius=1000,
        color='#d35400',
        fill=True
    ).add_child(folium.Popup(site_name)).add_to(site_map)

    # Add a label
    folium.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12; color:#d35400;"><b>{site_name}</b></div>'
        )
    ).add_to(site_map)

site_map

In [8]:
# Add a column 'marker_color' based on the launch outcome
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# Create MarkerCluster and add it to the site_map
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

# For each launch record, add a marker to the marker cluster
for index, row in spacex_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    outcome = 'Success' if row['class'] == 1 else 'Failure'
    marker_color = row['marker_color']

    marker = folium.Marker(
        location=coordinate,
        popup=f"Site: {row['Launch Site']}<br>Outcome: {outcome}",
        icon=folium.Icon(color=marker_color)
    )
    marker_cluster.add_child(marker)

site_map

In [9]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)

site_map

In [10]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [13]:
# Example for CCAFS LC-40
launch_site_lat = 28.563197
launch_site_lon = -80.576820
coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_coastline)
    )
)
site_map.add_child(distance_marker)

lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]], weight=2)
site_map.add_child(lines)

site_map

In [14]:
# Example for railway proximity to CCAFS LC-40
railway_lat = 28.57215
railway_lon = -80.58526
distance_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat, railway_lon)
railway_marker = folium.Marker(
    [railway_lat, railway_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_railway)
    )
)
site_map.add_child(railway_marker)
railway_line = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [railway_lat, railway_lon]], color='blue', weight=2)
site_map.add_child(railway_line)

site_map

In [15]:
# Example for highway proximity to CCAFS LC-40
highway_lat = 28.56301
highway_lon = -80.57089
distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)
highway_marker = folium.Marker(
    [highway_lat, highway_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_highway)
    )
)
site_map.add_child(highway_marker)
highway_line = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]], color='purple', weight=2)
site_map.add_child(highway_line)

site_map

In [16]:
# Example for city proximity (Titusville) to CCAFS LC-40
city_lat = 28.61222
city_lon = -80.80766
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)
city_marker = folium.Marker(
    [city_lat, city_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>{:10.2f} KM</b></div>'.format(distance_city)
    )
)
site_map.add_child(city_marker)
city_line = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [city_lat, city_lon]], color='orange', weight=2)
site_map.add_child(city_line)

site_map

In [17]:
folium.Marker([28.563197, -80.576820], popup='CCAFS LC-40', icon=folium.Icon(color='blue')).add_to(site_map)
folium.Marker([28.563197, -80.567000], 
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html=f'<div style="font-size: 12; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>'
    )
).add_to(site_map)
folium.PolyLine(locations=[[28.563197, -80.576820], [28.563197, -80.567000]], weight=2, color='red').add_to(site_map)
site_map